In [3]:
# Import packages.
import pandas as pd
import numpy as np
import os



In [4]:
# Load data.

# define file path, separate for organization.
netflix_prize_filepath = "/Users/calynguyen/Downloads/data/Netfli_prize/combined_data_1.txt"

# data is separated by "," so we can use pd.read_csv().
rows = []
movie_id = None

with open(netflix_prize_filepath, "r") as file:
    for line in file:
        line = line.strip()

        if line.endswith(":"):
            movie_id = line.replace(":", "")
        
        else:
            customer_id, movie_rating, date = line.split(",")
            rows.append([movie_id, customer_id, movie_rating, date ])
            
netflix_prize_data = pd.DataFrame(
    rows,
    columns = ["movie_id", "customer_id", "movie_rating", "date"]  
)

# Convert data to useable format.
netflix_prize_data["movie_id"] = netflix_prize_data["movie_id"].astype(int)
netflix_prize_data["customer_id"] = netflix_prize_data["customer_id"].astype(int)
netflix_prize_data["customer_movie_rating"] = netflix_prize_data["movie_rating"].astype(float)
netflix_prize_data["rating_date"] = pd.to_datetime(netflix_prize_data["date"])
netflix_prize_data = netflix_prize_data.drop(columns=["date","movie_rating"])

# Preview first couple rows.
netflix_prize_data.head()

,movie_id,customer_id,customer_movie_rating,rating_date
0,1,1488844,3.0,2005-09-06
1,1,822109,5.0,2005-05-13
2,1,885013,4.0,2005-10-19
3,1,30878,4.0,2005-12-26
4,1,823519,3.0,2004-05-03


In [5]:
# Feature Engineer.
netflix_prize_data["avg_customer_rating"] = netflix_prize_data.groupby("customer_id")["customer_movie_rating"].transform("mean")
netflix_prize_data["avg_movie_rating"] = netflix_prize_data.groupby("movie_id")["customer_movie_rating"].transform("mean")
netflix_prize_data["avg_date_rating"] = netflix_prize_data.groupby("rating_date")["customer_movie_rating"].transform("mean")
netflix_prize_data["count_of_ratings_per_date"] = netflix_prize_data.groupby("rating_date")["customer_movie_rating"].transform("count")
netflix_prize_data["count_of_ratings_per_movie"] = netflix_prize_data.groupby("movie_id")["customer_movie_rating"].transform("count")

# Preview.
netflix_prize_data.head()

,movie_id,customer_id,customer_movie_rating,rating_date,avg_customer_rating,avg_movie_rating,avg_date_rating,count_of_ratings_per_date,count_of_ratings_per_movie
0,1,1488844,3.0,2005-09-06,3.253308,3.749543,3.674945,45986,547
1,1,822109,5.0,2005-05-13,4.083333,3.749543,3.662106,30699,547
2,1,885013,4.0,2005-10-19,3.873563,3.749543,3.729616,40313,547
3,1,30878,4.0,2005-12-26,3.634304,3.749543,3.685429,11832,547
4,1,823519,3.0,2004-05-03,3.917197,3.749543,3.579395,27993,547


In [6]:
# Load Netflix movie titles.

# define file path, separate for organization.
netflix_titles_filepath = "/Users/calynguyen/Downloads/data/Netfli_prize/movie_titles.csv"

netflix_titles_data = pd.read_csv(
    netflix_titles_filepath,
    encoding="latin-1",
    header=None,
    names=["movie_id", "year", "title"],
    sep=",",
    engine="python",
    on_bad_lines="skip")

# Convert data to useable format.
netflix_titles_data["movie_id"] = netflix_titles_data["movie_id"].astype(int)
netflix_titles_data["release_year"] = pd.to_numeric(
    netflix_titles_data["year"],
    errors="coerce"
)
netflix_titles_data = netflix_titles_data.drop(columns="year")

netflix_titles_data["title"] = netflix_titles_data["title"].str.strip()

# checking duplicate titles returned "False", tells us we need to look further into the matter as movies can have duplicate titles with different release years.
netflix_titles_data["title"].is_unique
netflix_titles_data["movie_id"].is_unique

# Further inspection confirms that duplicate movies are released in separate years.
netflix_titles_data[
    netflix_titles_data["title"].duplicated(keep=False)
].sort_values("title")


# Preview first couple rows.
netflix_titles_data.head()

,movie_id,title,release_year
0,1,Dinosaur Planet,2003.0
1,2,Isle of Man TT 2004 Review,2004.0
2,3,Character,1997.0
3,4,Paula Abdul's Get Up & Dance,1994.0
4,5,The Rise and Fall of ECW,2004.0


In [7]:
netflix_merged_data = netflix_prize_data.merge(
    netflix_titles_data,
    on= "movie_id",
    how= "left"
)

netflix_merged_data.head()


,movie_id,customer_id,customer_movie_rating,rating_date,avg_customer_rating,avg_movie_rating,avg_date_rating,count_of_ratings_per_date,count_of_ratings_per_movie,title,release_year
0,1,1488844,3.0,2005-09-06,3.253308,3.749543,3.674945,45986,547,Dinosaur Planet,2003.0
1,1,822109,5.0,2005-05-13,4.083333,3.749543,3.662106,30699,547,Dinosaur Planet,2003.0
2,1,885013,4.0,2005-10-19,3.873563,3.749543,3.729616,40313,547,Dinosaur Planet,2003.0
3,1,30878,4.0,2005-12-26,3.634304,3.749543,3.685429,11832,547,Dinosaur Planet,2003.0
4,1,823519,3.0,2004-05-03,3.917197,3.749543,3.579395,27993,547,Dinosaur Planet,2003.0


In [14]:
# Feature engineer
netflix_merged_data["first_rating_date"] = (netflix_merged_data.groupby("customer_id")["rating_date"].transform("min"))
netflix_merged_data["last_rating_date"] = (netflix_merged_data.groupby("customer_id")["rating_date"].transform("max"))
netflix_merged_data["customer_active_days"] = (
    netflix_merged_data["last_rating_date"] - netflix_merged_data["first_rating_date"]
).dt.days

netflix_merged_data = netflix_merged_data.sort_values(["customer_id", "rating_date"])
netflix_merged_data["days_since_previous_rating"] = (
    netflix_merged_data.groupby("customer_id")["rating_date"].diff().dt.days
)
netflix_merged_data["count_of_ratings_per_customer"] = netflix_merged_data.groupby("customer_id")["customer_movie_rating"].transform("count")


netflix_merged_data["day_of_week"] = (
    netflix_merged_data["rating_date"].dt.day_name()
)

netflix_merged_data["is_weekend"] = (
    netflix_merged_data["rating_date"].dt.weekday >= 5
).astype(int)


netflix_merged_data["release_decade"] = (
    (netflix_merged_data["release_year"] // 10) * 10
)

netflix_merged_data["movie_age_at_rating"] = (
    (netflix_merged_data["rating_date"].dt.year) - netflix_merged_data["release_year"]
)

netflix_merged_data["avg_movie_rating_count_for_customer"] = (
    netflix_merged_data.groupby("customer_id")["count_of_ratings_per_movie"].transform("mean")
)

netflix_merged_data["customer_movie_rating_count_diff"] = (
    netflix_merged_data["count_of_ratings_per_movie"] - netflix_merged_data["avg_movie_rating_count_for_customer"]
)


netflix_merged_data.head()

,movie_id,customer_id,customer_movie_rating,rating_date,avg_customer_rating,avg_movie_rating,avg_date_rating,count_of_ratings_per_date,count_of_ratings_per_movie,title,...,last_rating_date,customer_active_days,days_since_previous_rating,count_of_ratings_per_customer,day_of_week,is_weekend,release_decade,movie_age_at_rating,avg_movie_rating_count_for_customer,customer_movie_rating_count_diff
3657676,705,6,3.0,2004-03-09,3.333333,3.640047,3.440096,20491,38680,Major League,...,2005-12-04,635,NaN,153,Tuesday,0,1980.0,15.0,54800.777778,-16120.777778
6495175,1267,6,3.0,2004-03-09,3.333333,3.118633,3.440096,20491,42113,Dr. Dolittle 2,...,2005-12-04,635,0.0,153,Tuesday,0,2000.0,3.0,54800.777778,-12687.777778
7800568,1561,6,3.0,2004-03-09,3.333333,3.480293,3.440096,20491,61019,American Wedding,...,2005-12-04,635,0.0,153,Tuesday,0,2000.0,1.0,54800.777778,6218.222222
10653731,2095,6,4.0,2004-03-09,3.333333,3.557507,3.440096,20491,87389,Liar Liar,...,2005-12-04,635,0.0,153,Tuesday,0,1990.0,7.0,54800.777778,32588.222222
12910599,2456,6,4.0,2004-03-09,3.333333,3.984098,3.440096,20491,19620,A Fistful of Dollars,...,2005-12-04,635,0.0,153,Tuesday,0,1960.0,40.0,54800.777778,-35180.777778
